# OC-NSFT with FAISS Accelerated Distance Computation

**Key improvement:** The `learn` function uses FAISS for fast nearest neighbor search.

**Performance gains:**
- `train_score`: O(n²) → O(n log n)
- `y_score`: O(n*m) → O(m log n)
- Memory: O(n²) → O(n)

**Note:** Full implementation is in `OC-NSFT_FAISS_learn.py`. Run `%run OC-NSFT_FAISS_learn.py` to load all functions.

In [1]:
import pandas as pd
import numpy as np
import faiss
from sklearn.metrics import (roc_auc_score, precision_score, average_precision_score,
                             recall_score, f1_score, accuracy_score, roc_curve, matthews_corrcoef)
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from scipy.sparse import csr_matrix as sp
from scipy.linalg import null_space
from scipy.spatial.distance import cdist
import time
import os

alpha = 0.9
print('FAISS loaded successfully')

FAISS loaded successfully


In [2]:
# ============================================================================
# FAISS UTILITIES - KEY OPTIMIZATION
# ============================================================================
def build_faiss_index(X, use_gpu=False):
    """Build FAISS index for fast nearest neighbor search."""
    X = np.ascontiguousarray(X.astype('float32'))
    d = X.shape[1]
    index = faiss.IndexFlatL2(d)
    if use_gpu and faiss.get_num_gpus() > 0:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(X)
    return index


def faiss_min_distance_train(X, k=2):
    """Compute min distance from each point to nearest neighbor (excluding self).
    Replaces O(n²) distance_vector computation with O(n log n)."""
    X = np.ascontiguousarray(X.astype('float32'))
    index = build_faiss_index(X)
    distances, _ = index.search(X, k)
    # FAISS returns squared L2 distances
    return np.sqrt(distances[:, 1])


def faiss_min_distance_test(X_test, X_train):
    """Compute min distance from each test point to nearest training point.
    Replaces O(n*m) minimum_distance computation with O(m log n)."""
    X_train = np.ascontiguousarray(X_train.astype('float32'))
    X_test = np.ascontiguousarray(X_test.astype('float32'))
    index = build_faiss_index(X_train)
    distances, _ = index.search(X_test, 1)
    return np.sqrt(distances[:, 0])

In [3]:
# ============================================================================
# DATA PREPROCESSING & CLUSTERING
# ============================================================================
def preprocess_data_noise(train_data, test_data, noise_percentage=10):
    print('..............................Data Overview................................')
    print('Train Data Shape:', train_data.shape)
    print('Test Data Shape:', test_data.shape)
    
    X_train_total = train_data.iloc[:, :-1].to_numpy()
    y_train_total = train_data.iloc[:, -1].to_numpy()
    X_train = X_train_total[y_train_total == 0]
    y_train = y_train_total[y_train_total == 0]
    
    n_samples = X_train.shape[0]
    noise_samples_count = int(n_samples * (noise_percentage / 100))
    X_train_noise = X_train_total[y_train_total == 1]
    noisy_indices = np.random.choice(X_train_noise.shape[0], size=noise_samples_count, replace=False)
    X_train_noise = X_train_noise[noisy_indices]
    
    X_train = np.vstack((X_train, X_train_noise))
    y_train = np.concatenate((y_train, np.ones(X_train_noise.shape[0])))
    
    X_test = test_data.iloc[:, :-1].to_numpy()
    y_test = test_data.iloc[:, -1].to_numpy()
    
    print('Number of samples after adding noise:', X_train.shape[0])
    return X_train, y_train, X_test, y_test


def cluster_kmeans(data, initial_k):
    print('Starting K-Means clustering...')
    kmeans = KMeans(n_clusters=initial_k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(data)
    sorted_indices = np.argsort(labels)
    print('Final number of clusters:', len(np.unique(labels)))
    return data[sorted_indices], labels[sorted_indices]

In [4]:
# ============================================================================
# NPD CALCULATION
# ============================================================================
def calculate_NPD(X, y, epsilon=1e-6):
    print('Begin calculating NPD and k --------------')
    X = X.T
    c = len(np.unique(y))
    d, N = X.shape
    
    t0 = time.time()
    mean_total = np.mean(X, axis=1, keepdims=True)
    P_t = X - mean_total
    
    P_w = np.zeros_like(X)
    for i in np.unique(y):
        class_mean = np.mean(X[:, y == i], axis=1, keepdims=True)
        P_w[:, y == i] = X[:, y == i] - class_mean
    
    S_w = np.dot(P_w, P_w.T) / N
    U, _, _ = np.linalg.svd(P_t, full_matrices=False)
    Q = U
    B = null_space(Q.T @ S_w @ Q)
    W = Q @ B
    
    t05 = time.time()
    print('Time train:', t05 - t0)
    print('W : d x L =', W.shape)
    return W, 0, (t05 - t0)

In [5]:
# ============================================================================
# LEARN FUNCTION - FAISS OPTIMIZED VERSION
# ============================================================================
def learn_faiss(npd, X_train, y_train, X_test):
    """FAISS-optimized learn function.
    
    Complexity improvement:
    - Original: O(n²) for train_score, O(n*m) for y_score
    - FAISS: O(n*log(n)) for train_score, O(m*log(n)) for y_score
    """
    null_point_X = (sp(X_train).dot(sp(npd))).toarray()
    null_point_X_test = (sp(X_test).dot(sp(npd))).toarray()

    t1 = time.time()
    
    # FAISS: O(n*log(n)) instead of O(n²)
    train_score = faiss_min_distance_train(null_point_X, k=2)
    
    # FAISS: O(m*log(n)) instead of O(n*m)
    y_score = faiss_min_distance_test(null_point_X_test, null_point_X)
    
    y_proba = np.zeros((len(y_score), 2))
    y_proba[:, 1] = np.minimum(y_score / np.max(train_score), 1)
    y_proba[:, 0] = 1 - y_proba[:, 1]
    y_proba = np.nan_to_num(y_proba, nan=1.0)
    y_predict = (y_proba[:, 1] > 0.2).astype(int)
    
    t2 = time.time()
    print('Time test (FAISS):', t2 - t1)
    return y_proba, y_predict, (t2 - t1)

# Default alias
learn = learn_faiss

In [6]:
# ============================================================================
# MODEL EVALUATION
# ============================================================================
def Model_evaluating(y_true, y_predict, y_scores):
    print('..............................Report Parameter...............................')
    y_prob = y_scores[:, 1]
    
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    print('Optimal threshold (Youden J):', optimal_threshold)

    y_predict_optimal = (y_prob >= optimal_threshold).astype(int)
    
    mcc = matthews_corrcoef(y_true, y_predict_optimal)
    f1 = f1_score(y_true, y_predict_optimal)
    ppv = precision_score(y_true, y_predict_optimal)
    recall = recall_score(y_true, y_predict_optimal)
    accuracy = accuracy_score(y_true, y_predict_optimal)
    auc_score = roc_auc_score(y_true, y_prob)
    aucpr = average_precision_score(y_true, y_prob)
    
    print(f'AUCROC: {auc_score*100:.2f}% | AUCPR: {aucpr*100:.2f}% | Acc: {accuracy*100:.2f}%')
    print(f'MCC: {mcc:.4f} | F1: {f1:.4f} | Prec: {ppv:.4f} | Recall: {recall:.4f}')
    return [auc_score*100, aucpr*100, accuracy*100, mcc, f1, ppv, recall]

In [7]:
# ============================================================================
# MAIN FUNCTION
# ============================================================================
columns = ['scaler', 'nCluster', 'noise_percentage', 'AUCROC', 'AUCPR', 'Accuracy',
           'MCC', 'F1 Score', 'Precision', 'Recall', 'Time Train', 'Time Test']

def function(df1, df2, scaler, noise, output_file, use_faiss=True):
    X_train0, y_train0, X_test, y_test = preprocess_data_noise(df1, df2, noise)
    
    imputer = SimpleImputer(strategy='mean')
    X_train0[np.isinf(X_train0)] = np.nan
    X_train0 = imputer.fit_transform(X_train0)
    
    learn_fn = learn_faiss if use_faiss else learn_faiss  # Can add learn_original for comparison
    
    for ncluster in range (1, 300, 20 ):
        X_train, y_train = cluster_kmeans(X_train0, ncluster)
        npd, k, training_time = calculate_NPD(X_train, y_train)
        y_proba, y_predict, inference_time = learn_fn(npd, X_train, y_train, X_test)
        v = Model_evaluating(y_test, y_predict, y_proba)
        
        result = [scaler, ncluster, noise] + v + [training_time, inference_time]
        result_df = pd.DataFrame([result], columns=columns)
        result_df.to_csv(output_file, mode='a', header=not os.path.exists(output_file), index=False)
    return 0

In [8]:
# ============================================================================
# EXPERIMENT RUNNER
# ============================================================================
dataset_prefixes = ['data_ToNIoT' ] # , 'data_CICIoT2023', 'data_N_BaIoT', 'data_BoTIoT']
scaler_names = ['StandardScaler', 'MinMaxScaler', 'Normalizer', 'QuantileTransformer', 'RobustScaler']

for prefix in dataset_prefixes:
    print('-' * 50)
    print('--------', prefix, '-' * 30)
    
    base_output_file = f'Results_FAISS_{prefix}'
    output_file = base_output_file + '_0.csv'
    counter = 0
    while os.path.exists(output_file):
        counter += 1
        output_file = f'{base_output_file}_{counter}.csv'
    
    for scaler in scaler_names:
        print(f'Processing {prefix} with {scaler}...')
        train_file = f'../../../Datascaled/NoiseOCData/Train_{scaler}_{prefix}.csv'
        test_file = f'../../../Datascaled/NoiseOCData/Test_{scaler}_{prefix}.csv'
        
        if not os.path.exists(train_file) or not os.path.exists(test_file):
            print('  Skipping: files not found')
            continue
        
        df_train = pd.read_csv(train_file).dropna()
        df_test = pd.read_csv(test_file).dropna()
        df_full = pd.concat([df_train, df_test], ignore_index=True)
        df_train_new, df_test_new = train_test_split(df_full, test_size=0.3, random_state=42)
        
        for noise in [0]:
            function(df_train_new, df_test_new, scaler, noise, output_file, use_faiss=True)

--------------------------------------------------
-------- data_ToNIoT ------------------------------
Processing data_ToNIoT with StandardScaler...
..............................Data Overview................................
Train Data Shape: (34149, 29)
Test Data Shape: (14636, 29)
Number of samples after adding noise: 3888
Starting K-Means clustering...
Final number of clusters: 1
Begin calculating NPD and k --------------
Time train: 0.14899420738220215
W : d x L = (28, 4)
Time test (FAISS): 0.16252446174621582
..............................Report Parameter...............................


/tmp/ipykernel_788992/3251832013.py:23: RuntimeWarning: divide by zero encountered in divide
  y_proba[:, 1] = np.minimum(y_score / np.max(train_score), 1)
/tmp/ipykernel_788992/3251832013.py:23: RuntimeWarning: invalid value encountered in divide
  y_proba[:, 1] = np.minimum(y_score / np.max(train_score), 1)
/home/jupyter-iclr2025/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Optimal threshold (Youden J): inf
AUCROC: 50.00% | AUCPR: 88.26% | Acc: 11.74%
MCC: 0.0000 | F1: 0.0000 | Prec: 0.0000 | Recall: 0.0000
Starting K-Means clustering...
Final number of clusters: 21
Begin calculating NPD and k --------------
Time train: 0.552401065826416
W : d x L = (28, 6)
Time test (FAISS): 0.13093018531799316
..............................Report Parameter...............................
Optimal threshold (Youden J): inf
AUCROC: 50.00% | AUCPR: 88.26% | Acc: 11.74%
MCC: 0.0000 | F1: 0.0000 | Prec: 0.0000 | Recall: 0.0000
Starting K-Means clustering...


/tmp/ipykernel_788992/3251832013.py:23: RuntimeWarning: divide by zero encountered in divide
  y_proba[:, 1] = np.minimum(y_score / np.max(train_score), 1)
/tmp/ipykernel_788992/3251832013.py:23: RuntimeWarning: invalid value encountered in divide
  y_proba[:, 1] = np.minimum(y_score / np.max(train_score), 1)
/home/jupyter-iclr2025/.local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Final number of clusters: 41
Begin calculating NPD and k --------------
Time train: 0.4184255599975586
W : d x L = (28, 10)
Time test (FAISS): 0.11550188064575195
..............................Report Parameter...............................
Optimal threshold (Youden J): 0.4552921652793884
AUCROC: 50.08% | AUCPR: 88.27% | Acc: 11.92%
MCC: 0.0111 | F1: 0.0042 | Prec: 0.9643 | Recall: 0.0021
Starting K-Means clustering...
Final number of clusters: 61
Begin calculating NPD and k --------------
Time train: 0.44342780113220215
W : d x L = (28, 12)
Time test (FAISS): 0.11712503433227539
..............................Report Parameter...............................
Optimal threshold (Youden J): 0.22262123227119446
AUCROC: 50.09% | AUCPR: 88.28% | Acc: 11.94%
MCC: 0.0122 | F1: 0.0046 | Prec: 0.9677 | Recall: 0.0023
Starting K-Means clustering...
Final number of clusters: 81
Begin calculating NPD and k --------------
Time train: 0.4824330806732178
W : d x L = (28, 12)
Time test (FAISS): 0.1068353